# Nightly SQM Measurements
**GOAL:**
<br> Plot nightly SQM measurements:
- *x-axis* = Time [sunset -> sunrise]
- *y-axis* = SQM 'mags'
- Filtere the data by cloudcover, lunar phase.

## 1) Read the data

In [113]:
import sqm_plot202X
import numpy as np
import datetime
import pandas as pd
import sys
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import plotly.express as px
import plotly.graph_objects as go

In [114]:
# User inputs:
loc = 'SPZN' # Available data: FCZN, DSZN, HILT, SPZN
year = '2020'
month = '6'

In [115]:
# Load the CSV
fdir = '/Users/jacksontobin/Local_Documents/NightTime_Research/FoCo Night Sky Team/SQM_DATA/SQM_COMPLETE'
# df_csv = f'{fdir}/{loc}_merged.csv'
df_csv = f'{fdir}/{loc}_combined.csv'
df = pd.read_csv(df_csv)
df = df.set_index(df['date'])
df.index = pd.to_datetime(df.index)

# Remove bogus data
cond = (df['mags'] < 0)
df['mags'] = df['mags'].mask(cond, np.nan)

# Filter the date by years
df_yr = df[(df.index < datetime.datetime(year=int(year)+1, month=1, day=1)) & 
           (df.index > datetime.datetime(year=int(year),   month=1, day=1))]

# Filter out data if the solar elevation is > -18 (astronomical twighlight cutoff)
df_yr.loc[(df_yr['sunalt_deg']) >= -18, 'mags'] = float('nan')


In [116]:
# Choose the dates
df_mo = df_yr[df_yr.index.month.isin([int(month)])]

## 3) Sort data by day?

### Filter by Lunar and Cloud data

In [117]:
# Define filtering conditions
phases = ['Full', 'Gibbous', 'Crescent', 'Quarter']
alts = [-5, -5, -5, -5]

# Remove data that meets these lunar phase conditions:
for phase, alt in zip(phases, alts):
    df_mo.loc[(df_mo['lunarphaseclass'] == phase) & (df_mo['lunar_alt'] >= alt), 'mags'] = float('nan')

# # Filter out cloud coverage
coverages = ['OVC', 'SCT', 'FEW', 'BKN', 'M']
df_mo.loc[df_mo['CC'].isin(coverages), 'mags'] = float('nan')

In [ ]:
# Convert the index to DateTime objects
df_mo.index = pd.to_datetime(df_mo.index)

# Convert to MST
# df_mo.index = df_mo.index.tz_localize('UTC').tz_convert('America/Denver')

df_mo

,Unnamed: 0.1,date,mags,sunalt_deg,lunar_alt,lunar_az,lunar_fraction,lunar_phase,lunarphaseclass,Unnamed: 0,cloudcover,cloudcover_low,cloudcover_mid,cloudcover_high,CC
date,,,,,,,,,,,,,,,
2020-06-01 01:00:00,3521189,2020-06-01 00:00:00,NaN,-25.463076,69.580961,197.601443,0.709682,0.334647,Gibbous,10752.0,100.0,0.0,100.0,97.0,OVC
2020-06-01 01:01:00,3521190,2020-06-01 00:01:00,NaN,-25.511237,69.793236,197.355871,0.709756,0.334647,Gibbous,10752.0,100.0,0.0,100.0,97.0,OVC
2020-06-01 01:02:00,3521191,2020-06-01 00:02:00,NaN,-25.558598,70.004774,197.110298,0.709830,0.334647,Gibbous,10752.0,100.0,0.0,100.0,97.0,OVC
2020-06-01 01:03:00,3521192,2020-06-01 00:03:00,NaN,-25.605155,70.215554,196.864712,0.709904,0.334647,Gibbous,10752.0,100.0,0.0,100.0,97.0,OVC
2020-06-01 01:04:00,3521193,2020-06-01 00:04:00,NaN,-25.650909,70.425534,196.619126,0.709978,0.334647,Gibbous,10752.0,100.0,0.0,100.0,97.0,OVC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-07-01 00:55:00,105544,2020-06-30 23:55:00,NaN,-23.955648,64.147040,205.769612,0.789688,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT
2020-07-01 00:56:00,105545,2020-06-30 23:56:00,NaN,-24.011855,64.382272,205.524559,0.789754,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT
2020-07-01 00:57:00,105546,2020-06-30 23:57:00,NaN,-24.067292,64.617401,205.279505,0.789819,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT


In [119]:

# Create a new column: all 8pm-6am periods fall on the same date
# df_mo['night'] = (df_mo.index - pd.Timedelta(hours=20)).normalize().tz_localize(None) # strips TZ for clean grouping
df_mo['night'] = np.where(df_mo.index.hour >= 12, df_mo.index.date, (df_mo.index - pd.Timedelta(days=1)).date)  # Adjust for times after midnight

hours = df_mo.index.hour + df_mo.index.minute / 60
hours = np.where(hours < 20, hours + 24, hours)  # Adjust hours to be in the range of 20-30 for 8pm-6am
df_mo['adjusted_hours'] = hours

print(df_mo['adjusted_hours'].min(), df_mo['adjusted_hours'].max())

# Filter only nighttime hours (8pm-6am)
t_start = 20 # 8pm
t_end = 6 #  6am
df_night = df_mo[((hours >= t_start) | (hours < t_end)) & df_mo['mags'].notna()].copy()

20.0 43.983333333333334


/var/folders/hx/hcs55bp10f5fdvcmjf0rmf4r0000gn/T/ipykernel_70673/2716145313.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/hx/hcs55bp10f5fdvcmjf0rmf4r0000gn/T/ipykernel_70673/2716145313.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



### Sort valid dates and assign colors

In [120]:

# Filter to a date range 
#   – Check sqm2020_interactive for reasonable data date ranges!
# df_night = df_night[(df_night['night'] >= t_start) & (df_night['night'] <= t_end)]

# Convert 'mags' to candela per square meter
df_night['candela'] = 108000*10**(-0.4*df_night['mags'])

# Get the Timestamp of nights
nights = sorted(df_night['night'].unique()) # list of sorted nights (Timestamps)

# Assign colors for each night
cmap = cm.brg
colors = cmap(np.linspace(0, 1, len(nights)))
night_color = {night: color for night, color in zip(nights, colors)}

### Plot the data

In [121]:
from numpy.polynomial import polynomial as P

plot_bestfit = False
plot_scatter = True

fig = go.Figure()

for night, group in df_night.groupby('night'):
    
    # Use fractional hours for x-axis, wrapping past-midnight ti000mes
    # hours = group.index.hour + group.index.minute / 60
    # hours = np.where(hours < t_start, hours + 24, hours)  # 1am -> 25, etc

    night_str = night.strftime('%Y-%m-%d')

    if plot_scatter:
        fig.add_trace(
            go.Scatter(
                x=group['adjusted_hours'], 
                y=group['candela'], 
                name=night_str, 
                mode='markers',
                marker=dict(
                    size=8,
                ),
                ))
    
    # Add a best-fit line with variance:
    if plot_bestfit:
        if len(hours) < 6:
            continue

        # Sort by hour for a clean line
        sort_idx = np.argsort(hours)
        hours_sorted = hours[sort_idx]
        mags_sorted = group['candela'][sort_idx]

        # Fit 5th degree polynomial
        coeffs = np.polyfit(hours_sorted, mags_sorted, 4)
        poly = np.poly1d(coeffs)

        # Evaluate on a smooth grid
        x_grid = np.linspace(hours_sorted.min(), hours_sorted.max(), 300)
        y_fit = poly(x_grid)

        # Residual std dev as variance estimate
        y_at_data = poly(hours_sorted)
        residuals = mags_sorted - y_at_data
        std = np.std(residuals)

        color = night_color[night]
        ax.plot(x_grid, y_fit, color=color, linewidth=1.5, label=str(night.date()))
        ax.fill_between(x_grid, y_fit - std, y_fit + std, color=color, alpha=0.15)

fig.update_layout(
    title=dict(
        text=f'SQM Measurements: {loc}, {t_start} to {t_end}'
    ),
    xaxis=dict(
        title=dict(text='Hour'),
        tickmode='array',
        tickvals=[20,21, 22, 23, 24, 25,26,27,28,29,30],
        ticktext=['8pm','9pm','10pm','11pm','12am','1am','2am','3am','4am','5am','6am']
    ),
    yaxis=dict(
        title=dict(text='Cd/m^2')
    ),
)

fig.show()
fig.write_html(f'./{loc}_{t_start}_{t_end}_interactive.html')